# Day 5 — Hands-On Lab 2: Build the Silver Layer — All Sources

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Sources** | `gbmart.bronze.{customers, products, addresses, orders, order_items, payments, payment_methods}` |
| **Targets** | matching `gbmart.silver.*` tables |
| **Duration** | ~3 hours |
| **Follows** | HOL 1 (you already built `silver.customers` there) |

### Learning Objectives
- Apply the standardize/conform/enrich + DQ/quarantine pattern consistently across every source
- Recognize when a table needs SCD2 (dimensions that change) vs. SCD1 / plain overwrite (facts, reference data)
- Handle a semi-structured source (`products` has nested structs and an array column)
- Run referential-integrity checks *across* Silver tables, not just within one

### The pattern, once, for reference

| Step | What happens |
|---|---|
| 1 | Read Bronze, inspect schema |
| 2 | DQ scan — tag every row with its first failing rule |
| 3 | Investigate flagged rows — don't fix blind |
| 4 | Fix / flag / quarantine — pick correctly, see Day 5 ILT 2 |
| 5 | Transform — standardize, conform, enrich |
| 6 | Write to Silver (+ quarantine table if needed) |
| 7 | Verify |

---
**Instructions:** Work through each source in order — some later sources (`order_items`, `payments`) check referential integrity against Silver tables you build earlier in this same notebook, so don't skip ahead.

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark.conf.set("spark.sql.ansi.enabled", "false")

CATALOG = "gbmart"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
print(f"Schema '{CATALOG}.silver' ready.")

---
## Source 1 — Customers (already built in HOL 1)

You built `gbmart.silver.customers` and `gbmart.silver.customers_quarantine` in HOL 1, applying: email fix, phone standardization, REGISTERED_UNDER_18 quarantine, `NOT NULL`/`CHECK` constraints, and SCD2 columns (`customer_sk`, `is_current`, `effective_start_date`, `effective_end_date`) — customers is a dimension whose attributes (email, phone) can change over time, so it needs full SCD2 tracking, not just an overwrite. Confirm it's there before moving on.

In [ ]:
assert spark.catalog.tableExists("gbmart.silver.customers"), "Run HOL 1 first — silver.customers must exist."
print(f"silver.customers : {spark.table('gbmart.silver.customers').count():,} rows")
print(f"is_current split : ")
spark.table("gbmart.silver.customers").groupBy("is_current").count().show()

---
## Source 2 — Products (semi-structured, SCD2 initial load)

`bronze.products` isn't flat — alongside plain columns it carries two **struct** columns (`specs`, `supplier_info`) and one **array** column (`tags`). Inspect the real shape before assuming anything.

In [ ]:
bronze_products = spark.table("gbmart.bronze.products")
print(f"Total records: {bronze_products.count():,}")
bronze_products.printSchema()

### Check for silent schema drift

Autoloader writes any field it couldn't match to the inferred schema into `_rescued_data` instead of silently dropping it. Keep this check permanently — it's how a future file with a renamed/extra field would surface.

In [ ]:
rescued_count = bronze_products.filter(col("_rescued_data").isNotNull()).count()
print(f"Rows with non-null _rescued_data: {rescued_count:,} / {bronze_products.count():,}")

### Flatten `specs` / `supplier_info`; keep `tags` as an array

`tags` has a constant cardinality across every product (check it yourself with `size("tags")` grouped) — that means it's a simple multi-valued *attribute*, not a many-to-many relationship, so it stays `array<string>` rather than being exploded into a bridge table.

In [ ]:
flattened_products = bronze_products.select(
    "product_id", "product_name", "category", "sub_category",
    "actual_price_inr", "discounted_price_inr", "rating", "num_ratings", "last_updated",

    col("specs.power_source").alias("power_source"),
    col("specs.country_of_origin").alias("country_of_origin"),
    col("specs.color_options").alias("color_options"),
    col("specs.is_returnable").alias("is_returnable"),
    col("specs.return_window_days").alias("return_window_days"),
    col("specs.material").alias("material"),          # NULL by category design (electronics have no 'material') -- not a DQ issue
    col("specs.warranty_months").alias("warranty_months"),
    col("specs.weight_kg").alias("weight_kg"),

    col("supplier_info.supplier_id").alias("supplier_id"),
    col("supplier_info.name").alias("supplier_name"),
    col("supplier_info.city").alias("supplier_city"),

    "tags", "_source_file"
)
flattened_products.printSchema()

### DQ scan — products

In [ ]:
dq_scan_products = flattened_products.withColumn("_dq_issue",
    when(col("product_id").isNull(),                                          lit("NULL_PRODUCT_ID"))
    .when(col("product_name").isNull(),                                       lit("NULL_PRODUCT_NAME"))
    .when(col("category").isNull(),                                           lit("NULL_CATEGORY"))
    .when(col("actual_price_inr").isNull() | (col("actual_price_inr") <= 0),   lit("INVALID_PRICE"))
    .when(col("discounted_price_inr") > col("actual_price_inr"),               lit("DISCOUNT_EXCEEDS_PRICE"))
    .when(~col("rating").between(0, 5),                                        lit("INVALID_RATING"))
    .when(col("supplier_id").isNull(),                                        lit("NULL_SUPPLIER_ID"))
    .otherwise(lit(None))
)

print("=== DQ Issues Found ===")
dq_scan_products.groupBy("_dq_issue").count().orderBy("count", ascending=False).show()
# products.csv is well-formed enough that this typically comes back with 0 issues --
# no quarantine table needed for this source.

### Transform + write — SCD2 initial load

Every product gets its **first** SCD2 version here (`is_current = True`, open-ended `effective_end_date`). The `MERGE`-based logic that closes out an old price version when a price changes is built later in the course (Incremental Loading & SCD, Day 10) — not here.

In [ ]:
silver_products = (
    dq_scan_products.filter(col("_dq_issue").isNull()).drop("_dq_issue")
    .withColumn("discount_pct",
        round((col("actual_price_inr") - col("discounted_price_inr")) / col("actual_price_inr") * 100, 2)
    )
    .withColumn("effective_start_date", current_date())
    .withColumn("effective_end_date", lit(None).cast(DateType()))
    .withColumn("is_current", lit(True))
    # Surrogate key -- generated once here in Silver, reused by every Gold table
    .withColumn("product_sk", sha2(concat_ws("|", col("product_id"), col("effective_start_date").cast("string")), 256))
    .withColumn("_silver_updated_at", current_timestamp())
)

silver_products.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gbmart.silver.products")

print(f"silver.products : {spark.table('gbmart.silver.products').count():,} rows")

---
## Source 3 — Address (recover a corrupted value from a duplicate copy elsewhere in the row)

`PinCode` was inferred as numeric by Autoloader — any PIN code that originally started with `0` already lost that digit before Spark ever showed it to us. But `AddressLine1` embeds the same PinCode as plain text (e.g. `Guntur_082130`), and text never strips leading zeros. That gives an independent, trustworthy second copy — no guessing required.

In [ ]:
bronze_address = spark.table("gbmart.bronze.addresses")

pincode_check = bronze_address.withColumn("_pincode_len", length(col("PinCode").cast("long").cast("string")))
short_count = pincode_check.filter(col("_pincode_len") < 6).count()
print(f"Rows with PinCode shorter than 6 digits: {short_count:,} / {bronze_address.count():,}")

In [ ]:
# Recover PinCode from AddressLine1 where it's short; leave the rest untouched
recovered_address = bronze_address.withColumn(
    "_extracted_pincode", regexp_extract(col("AddressLine1"), r"(\d{5,6})\s*$", 1)
).withColumn(
    "_pincode_len", length(col("PinCode").cast("long").cast("string"))
).withColumn(
    "PinCode",
    when(col("_pincode_len") < 6, col("_extracted_pincode"))
    .otherwise(col("PinCode").cast("long").cast("string"))
).drop("_extracted_pincode", "_pincode_len")

# Every PinCode should now be exactly 6 characters
recovered_address.select(length(col("PinCode")).alias("len")).groupBy("len").count().show()

### Remove the duplicate City/PinCode text embedded in `AddressLine1`

`AddressLine1` embeds the street address **plus** a duplicate of City/PinCode at the end, joined with inconsistent separators (`\n`, `_`, comma). Since `City`/`PinCode` already exist as clean columns, this trailing chunk is pure duplication — remove everything after the *last* comma or line break.

In [ ]:
cleaned_address = recovered_address \
    .withColumn("AddressLine1", regexp_replace(col("AddressLine1"), r"[,\n][^,\n]*$", "")) \
    .withColumn("AddressLine1", regexp_replace(col("AddressLine1"), r"\n", ", ")) \
    .withColumn("AddressLine1", regexp_replace(col("AddressLine1"), r"_", " ")) \
    .withColumn("AddressLine1", trim(col("AddressLine1"))) \
    .withColumn("State", regexp_replace(col("State"), r"_", " ")) \
    .withColumn("AddressType", regexp_replace(col("AddressType"), r"_", " "))

cleaned_address.select("AddressID", "AddressLine1", "City", "PinCode", "State", "AddressType").show(10, truncate=False)

In [ ]:
# DQ scan -- nulls + AddressID uniqueness. This source needs no quarantine table
# (address is a straightforward reference table once the two fixes above are applied).
dupe_count = cleaned_address.groupBy("AddressID").count().filter("count > 1").count()
print(f"Duplicate AddressID count: {dupe_count}")

# NOTE: table name is singular "address" (not "addresses") -- matches the real
# gbmart workspace convention that Day 6/7's Gold build already reads from.
silver_address = cleaned_address \
    .withColumnRenamed("AddressID", "address_id") \
    .withColumnRenamed("CustomerID", "customer_id") \
    .withColumnRenamed("AddressLine1", "address_line1") \
    .withColumnRenamed("City", "city") \
    .withColumnRenamed("State", "state") \
    .withColumnRenamed("PinCode", "pincode") \
    .withColumnRenamed("AddressType", "address_type") \
    .withColumn("_silver_updated_at", current_timestamp()) \
    .select("address_id", "customer_id", "address_line1", "city", "state", "pincode", "address_type", "_source_file", "_silver_updated_at")

silver_address.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gbmart.silver.address")

print(f"silver.address : {spark.table('gbmart.silver.address').count():,} rows")

---
## Source 4 — Orders (Lakeflow Connect CDC, SCD1 MERGE)

Orders are ingested via **Lakeflow Connect CDC** — Bronze holds the latest snapshot per order, and Lakeflow **lowercases every column name** (`OrderID` -> `orderid`). An order has one current truth at any moment (its status, its expected delivery), so this is **SCD1**: `MERGE` overwrites matched rows, inserts new ones — no history kept, unlike customers/products.

In [ ]:
bronze_orders = spark.table("gbmart.bronze.orders")
VALID_CHANNELS = ["Online", "Retail PoS"]
print(f"Total records: {bronze_orders.count():,}")
print(f"Columns: {bronze_orders.columns}")

### The DELIVERY_BEFORE_SHIP decision (see Day 5 ILT 2 for the full investigation)

**Do not quarantine these rows.** The full investigation (ILT 2) found the 114 flagged orders are a timezone-cast artifact, not real data errors — quarantining them previously orphaned 352 downstream `order_items`. We keep every order and add a `_data_note` flag instead.

In [ ]:
transformed_orders = (
    bronze_orders
    .withColumn("orderdate",            col("orderdate").cast(DateType()))
    .withColumn("shippingdate",         col("shippingdate").cast(DateType()))
    .withColumn("expecteddeliverydate", col("expecteddeliverydate").cast(DateType()))
    .withColumn("actualdeliverydate",   col("actualdeliverydate").cast(DateType()))

    .withColumn("order_to_ship_days",
        when(col("shippingdate").isNotNull(), datediff(col("shippingdate"), col("orderdate"))).otherwise(lit(None).cast("int")))
    .withColumn("ship_to_delivery_days",
        when(col("actualdeliverydate").isNotNull() & col("shippingdate").isNotNull(),
             datediff(col("actualdeliverydate"), col("shippingdate"))).otherwise(lit(None).cast("int")))
    .withColumn("delivery_delay_days",
        when(col("actualdeliverydate").isNotNull() & col("expecteddeliverydate").isNotNull(),
             datediff(col("actualdeliverydate"), col("expecteddeliverydate"))).otherwise(lit(None).cast("int")))
    .withColumn("is_delivered", col("actualdeliverydate").isNotNull())
    .withColumn("is_late",
        when(col("actualdeliverydate").isNotNull() & col("expecteddeliverydate").isNotNull(),
             col("actualdeliverydate") > col("expecteddeliverydate")).otherwise(lit(None).cast("boolean")))
    .withColumn("order_status",
        when(col("actualdeliverydate").isNotNull(), lit("Delivered"))
        .when(col("shippingdate").isNotNull(),      lit("Shipped"))
        .otherwise(                                  lit("Pending")))
    .withColumn("_data_note",
        when(
            col("actualdeliverydate").isNotNull() & col("shippingdate").isNotNull() &
            (col("actualdeliverydate") < col("shippingdate")),
            lit("POSSIBLE_TIMEZONE_OFFSET_1DAY")
        ).otherwise(lit(None).cast("string")))

    .withColumnRenamed("orderid", "order_id").withColumnRenamed("customerid", "customer_id")
    .withColumnRenamed("orderdate", "order_date").withColumnRenamed("shippingdate", "shipping_date")
    .withColumnRenamed("expecteddeliverydate", "expected_delivery_date")
    .withColumnRenamed("actualdeliverydate", "actual_delivery_date")
    .withColumnRenamed("shippingtierid", "shipping_tier_id").withColumnRenamed("supplierid", "supplier_id")
    .withColumnRenamed("orderchannel", "order_channel")
)

silver_orders_df = transformed_orders.withColumn("_silver_updated_at", current_timestamp()).select(
    "order_id", "customer_id", "order_date", "order_channel", "order_status",
    "shipping_tier_id", "supplier_id", "shipping_date", "expected_delivery_date", "actual_delivery_date",
    "order_to_ship_days", "ship_to_delivery_days", "delivery_delay_days",
    "is_delivered", "is_late", "_data_note", "updated_at", "_silver_updated_at"
)
print(f"Transformed rows: {silver_orders_df.count():,}")
print(f"Rows with _data_note flag: {silver_orders_df.filter(col('_data_note').isNotNull()).count():,}")

In [ ]:
# SCD1 write: MERGE keyed on order_id. Matched -> overwrite all columns (CDC told
# us the order changed). Unmatched -> insert (new order placed). Safe to re-run.
SILVER_ORDERS = "gbmart.silver.orders"

if not spark.catalog.tableExists(SILVER_ORDERS):
    silver_orders_df.write.format("delta").saveAsTable(SILVER_ORDERS)
    print(f"Created {SILVER_ORDERS} — initial load, {silver_orders_df.count():,} rows")
else:
    tgt = DeltaTable.forName(spark, SILVER_ORDERS)
    (tgt.alias("tgt")
        .merge(silver_orders_df.alias("src"), "tgt.order_id = src.order_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print(f"Merged into {SILVER_ORDERS} — SCD1 upsert complete")

print(f"{SILVER_ORDERS} row count: {spark.table(SILVER_ORDERS).count():,}")

---
## Source 5 — Order Items (grain of the eventual `fact_sales`)

No derived columns needed here — pricing/totals get computed when joined with `products` at Gold. The important step is **referential integrity**: every `order_id`/`product_id` here must exist in the Silver tables you already built above.

In [ ]:
bronze_order_items = spark.table("gbmart.bronze.order_items")

dq_scan_items = bronze_order_items.withColumn("_dq_issue",
    when(col("orderitemid").isNull(),                        lit("NULL_ORDER_ITEM_ID"))
    .when(col("orderid").isNull(),                           lit("NULL_ORDER_ID"))
    .when(col("productid").isNull(),                         lit("NULL_PRODUCT_ID"))
    .when(col("quantity").isNull() | (col("quantity") <= 0), lit("INVALID_QUANTITY"))
    .otherwise(lit(None))
)
print("=== DQ Issues Found ===")
dq_scan_items.groupBy("_dq_issue").count().orderBy("count", ascending=False).show()

In [ ]:
# Referential integrity: left_anti join finds any order_item whose parent doesn't exist
orders_df   = spark.table("gbmart.silver.orders")
products_df = spark.table("gbmart.silver.products")

orphan_orders   = bronze_order_items.join(orders_df,   bronze_order_items.orderid   == orders_df.order_id,   "left_anti")
orphan_products = bronze_order_items.join(products_df, bronze_order_items.productid == products_df.product_id, "left_anti")

print(f"order_items with no matching order_id  : {orphan_orders.count():,} / {bronze_order_items.count():,}")
print(f"order_items with no matching product_id: {orphan_products.count():,} / {bronze_order_items.count():,}")
print("Both should be 0 -- if not, revisit the orders quarantine/flag decision above before proceeding.")

In [ ]:
silver_order_items = bronze_order_items \
    .withColumnRenamed("orderitemid", "order_item_id") \
    .withColumnRenamed("orderid", "order_id") \
    .withColumnRenamed("productid", "product_id") \
    .withColumn("_silver_updated_at", current_timestamp()) \
    .select("order_item_id", "order_id", "product_id", "quantity", "updated_at", "_silver_updated_at")

silver_order_items.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gbmart.silver.order_items")

print(f"silver.order_items : {spark.table('gbmart.silver.order_items').count():,} rows")

---
## Source 6 — Payments (conditional-null business rule)

`GiftCardAmount`/`CouponAmount` are only populated when the matching `*Usage` flag is `"Yes"`. Before assuming that's a DQ gap, confirm it's actually a clean, consistent business rule.

In [ ]:
bronze_payments = spark.table("gbmart.bronze.payments")

print("=== GiftCard: Usage vs Amount nullness ===")
bronze_payments.groupBy("GiftCardUsage").agg(
    count(when(col("GiftCardAmount").isNull(), 1)).alias("amount_null"),
    count(when(col("GiftCardAmount").isNotNull(), 1)).alias("amount_populated")
).show()
# Expected: Usage='No' -> amount always null; Usage='Yes' -> amount always populated.
# If that holds with 0 violations, it's a clean business rule, not a DQ issue.

In [ ]:
dq_scan_payments = bronze_payments.withColumn("_dq_issue",
    when(col("PaymentID").isNull(),                 lit("NULL_PAYMENT_ID"))
    .when(col("OrderID").isNull(),                  lit("NULL_ORDER_ID"))
    .when(col("PaymentDate").isNull(),               lit("NULL_PAYMENT_DATE"))
    .when(col("PaymentDate") > current_timestamp(),  lit("FUTURE_PAYMENT_DATE"))
    .when(col("PaymentMethodID").isNull(),           lit("NULL_PAYMENT_METHOD"))
    .otherwise(lit(None))
)
dq_scan_payments.groupBy("_dq_issue").count().orderBy("count", ascending=False).show()

# Referential integrity against silver.orders (built above)
orphan_payment_orders = bronze_payments.join(
    spark.table("gbmart.silver.orders").select("order_id"),
    bronze_payments.OrderID == col("order_id"), "left_anti"
)
print(f"Payments with no matching order_id: {orphan_payment_orders.count():,} / {bronze_payments.count():,}")

In [ ]:
silver_payments = bronze_payments \
    .withColumn("used_any_discount", (col("GiftCardUsage") == "Yes") | (col("CouponUsage") == "Yes")) \
    .withColumnRenamed("PaymentID", "payment_id").withColumnRenamed("OrderID", "order_id") \
    .withColumnRenamed("PaymentDate", "payment_date") \
    .withColumnRenamed("GiftCardUsage", "gift_card_usage").withColumnRenamed("GiftCardAmount", "gift_card_amount") \
    .withColumnRenamed("CouponUsage", "coupon_usage").withColumnRenamed("CouponAmount", "coupon_amount") \
    .withColumnRenamed("PaymentMethodID", "payment_method_id") \
    .withColumn("_silver_updated_at", current_timestamp()) \
    .select("payment_id", "order_id", "payment_date", "payment_method_id",
            "gift_card_usage", "gift_card_amount", "coupon_usage", "coupon_amount",
            "used_any_discount", "_source_file", "_silver_updated_at")

silver_payments.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gbmart.silver.payments")

print(f"silver.payments : {spark.table('gbmart.silver.payments').count():,} rows")

---
## Source 7 — Payment Methods (small lookup table)

A small reference table (`PaymentMethodID` -> `MethodName`) — this is exactly what `dim_payment_method` needs in Gold, and what `silver.payments` would join against to display `"Credit Card"` instead of `PM-001`. Unlike the other 6 sources, Day 4 never ingested this one into Bronze, so the next cell builds `gbmart.bronze.payment_methods` first, using the exact same Autoloader pattern as everything else in this course.

In [ ]:
# ─── Bronze catch-up: payment_methods (Day 4 never built this table) ──────────
# Day 4 HOL 1 built exactly 6 Bronze tables: customers, addresses, payments,
# products (Autoloader) + orders, order_items (CDC). payment_methods was not
# among them -- but silver.payments needs it for the payment-method conform
# join, and dim_payment_method will need it in Gold. We build it here using
# the IDENTICAL Autoloader pattern Day 4 used for customers/addresses/payments.
#
# ASSUMPTION: raw-data/payment_methods/ is a sibling folder to raw-data/customers/,
# raw-data/addresses/, raw-data/payments/ under the same external location. If your
# workspace laid out payment_methods.csv differently, adjust PM_SOURCE_PATH below.

EXTERNAL_LOCATION = "abfss://ecom-gbmart-data@ecomadlsdata.dfs.core.windows.net/raw-data"

PM_SOURCE_PATH     = f"{EXTERNAL_LOCATION}/payment_methods/"
PM_CHECKPOINT_PATH = f"{EXTERNAL_LOCATION}/_checkpoints/payment_methods/"
PM_SCHEMA_PATH     = f"{EXTERNAL_LOCATION}/_schemas/payment_methods/"

payment_methods_bronze_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",              "csv")
    .option("cloudFiles.schemaLocation",      PM_SCHEMA_PATH)
    .option("cloudFiles.inferColumnTypes",    "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("header",                         "true")
    .load(PM_SOURCE_PATH)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

(
    payment_methods_bronze_df.writeStream
    .format("delta")
    .outputMode("append")                          # same append-only rule as every other Bronze table
    .option("checkpointLocation", PM_CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("gbmart.bronze.payment_methods")
)

print(f"gbmart.bronze.payment_methods : {spark.table('gbmart.bronze.payment_methods').count():,} rows")
spark.table("gbmart.bronze.payment_methods").show(truncate=False)

In [ ]:
bronze_payment_methods = spark.table("gbmart.bronze.payment_methods")

dupe_count = bronze_payment_methods.groupBy("PaymentMethodID").count().filter("count > 1").count()
print(f"Duplicate PaymentMethodID count: {dupe_count}  (expected: 0)")

silver_payment_methods = bronze_payment_methods \
    .withColumnRenamed("PaymentMethodID", "payment_method_id") \
    .withColumnRenamed("MethodName", "method_name") \
    .withColumn("_silver_updated_at", current_timestamp()) \
    .select("payment_method_id", "method_name", "_source_file", "_silver_updated_at")

silver_payment_methods.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gbmart.silver.payment_methods")

print(f"silver.payment_methods : {spark.table('gbmart.silver.payment_methods').count():,} rows")

---
## Final Verification — Row Counts Across the Whole Silver Layer

In [ ]:
silver_tables = [
    "customers", "products", "address", "orders",
    "order_items", "payments", "payment_methods"
]

for t in silver_tables:
    full_name = f"gbmart.silver.{t}"
    count = spark.table(full_name).count()
    print(f"{full_name:35s} : {count:>10,} rows")

**Q: Which two tables use SCD2 (`is_current`/`effective_start_date`/`effective_end_date`), and why do they need it while `orders` and `payments` don't?**

*Your answer:* _______________

---
## Submission Checklist

```
Submission Checklist
----------------------------------------------------------
[ ] silver.customers        -- confirmed from HOL 1 (incl. NOT NULL/CHECK constraints)
[ ] silver.products         -- flattened structs, tags kept as array, SCD2 initial load
[ ] silver.address          -- PinCode recovered from AddressLine1, duplication removed
[ ] silver.orders           -- SCD1 MERGE, DELIVERY_BEFORE_SHIP flagged (not quarantined)
[ ] silver.order_items      -- 0 orphans against orders and products
[ ] silver.payments         -- GiftCard/Coupon business rule confirmed, 0 orphans
[ ] silver.payment_methods  -- bronze.payment_methods built first, 0 duplicate IDs
── Total rows across all 7 Silver tables: ______
----------------------------------------------------------
```